# ⚔️ CRUSADER — F03 SIGISMUND (Modal GPU)
## Rendu Remotion distribué → short_render.mp4

> *"Sigismund stood unmoved, and from him radiated the Emperor's will."*

---

**Ce notebook utilise Modal pour le rendu GPU distribué.**  
**Temps estimé : ~15-20 min (3 workers A10G en parallèle)**

### Étapes :
1. Montage Google Drive
2. Installation Modal
3. Authentification Modal (token)
4. Téléchargement du worker Modal depuis GitHub
5. Configuration des chemins
6. Validation CUSTOS check-out
7. Upload assets vers Modal Volume
8. Dispatch rendu parallèle (N workers GPU)
9. Concat chunks → short_render.mp4
10. Sauvegarde sur Drive (F03/OUT/)
11. Validation CUSTOS check-in
12. Aperçu et téléchargement

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Installation Modal

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'modal', '-q'], check=True)
import modal
print(f'[OK] Modal {modal.__version__} installé')

---
## Étape 3 — Authentification Modal

> Récupérez votre `Token ID` et `Token Secret` sur **https://modal.com/settings/tokens**
>
> Collez-les dans les deux champs ci-dessous.
>
> **Option recommandée** : ajoutez `MODAL_TOKEN_ID` et `MODAL_TOKEN_SECRET`
> dans l'onglet Secrets Colab (icône clé à gauche), puis décommentez Option A.

In [ ]:
import os

# ── OPTION A : Secrets Colab (recommandé) ──────────────────────────────────
# from google.colab import userdata
# os.environ['MODAL_TOKEN_ID']     = userdata.get('MODAL_TOKEN_ID')
# os.environ['MODAL_TOKEN_SECRET'] = userdata.get('MODAL_TOKEN_SECRET')

# ── OPTION B : Saisie directe ───────────────────────────────────────────────
os.environ['MODAL_TOKEN_ID']     = 'COLLER_VOTRE_TOKEN_ID_ICI'
os.environ['MODAL_TOKEN_SECRET'] = 'COLLER_VOTRE_TOKEN_SECRET_ICI'

# ── Vérification ───────────────────────────────────────────────────────────
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'modal', 'profile', 'current'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f'[OK] Modal authentifié — {r.stdout.strip()}')
else:
    print('[ERREUR] Token invalide. Vérifiez MODAL_TOKEN_ID et MODAL_TOKEN_SECRET.')
    print(r.stderr)

---
## Étape 4 — Téléchargement du worker Modal et de CUSTOS depuis GitHub

In [ ]:
import urllib.request, os, subprocess, sys

REPO_RAW    = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
SCRIPTS_DIR = '/content/crusader_scripts'
os.makedirs(SCRIPTS_DIR, exist_ok=True)

files_to_download = [
    ('F03_SIGISMUND/CODEBASE/crs_f03_modal_worker.py', 'crs_f03_modal_worker.py'),
    ('CRS_CUSTOS.py',                                   'CRS_CUSTOS.py'),
]

for rel_path, dest_name in files_to_download:
    dest_path = os.path.join(SCRIPTS_DIR, dest_name)
    urllib.request.urlretrieve(f'{REPO_RAW}/{rel_path}', dest_path)
    print(f'[OK] {dest_name}')

print('\nScripts téléchargés.')

# ── Déploiement de l'app Modal (requis pour fire-and-forget) ──────────────
print('\n[DEPLOY] Déploiement de l\'app Modal (première fois : ~5-8 min, ensuite rapide)...')
deploy_result = subprocess.run(
    [sys.executable, '-m', 'modal', 'deploy',
     os.path.join(SCRIPTS_DIR, 'crs_f03_modal_worker.py')],
    capture_output=True, text=True, cwd=SCRIPTS_DIR
)
if deploy_result.returncode == 0:
    print('[OK] App Modal déployée — workers indépendants du client Colab.')
else:
    print('[ERREUR] Deploy Modal échoué.')
    print(deploy_result.stderr[-2000:])


---
## Étape 5 — Configuration des chemins

> **Modifiez `DRIVE_BASE` et `N_WORKERS` si nécessaire.**

In [ ]:
import os

# ── MODIFIEZ ICI SI NÉCESSAIRE ─────────────────────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/DRIVE_CRUSADER'
N_WORKERS   = 3      # Nombre de workers GPU parallèles (1-5 selon budget $5)
COMPOSITION = 'CrusaderShort'
# ──────────────────────────────────────────────────────────────────────────────

SCRIPTS_DIR = '/content/crusader_scripts'
F03_IN      = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'IN')
F03_OUT     = os.path.join(DRIVE_BASE, 'F03_SIGISMUND', 'OUT')
os.makedirs(F03_OUT, exist_ok=True)

print('Configuration :')
print(f'  F03 IN   : {F03_IN}')
print(f'  F03 OUT  : {F03_OUT}')
print(f'  Workers  : {N_WORKERS}')
print()

checks = {
    'timing.json':     os.path.isfile(os.path.join(F03_IN, 'timing.json')),
    'roadmap.json':    os.path.isfile(os.path.join(F03_IN, 'roadmap.json')),
    'audio_clean.mp3': os.path.isfile(os.path.join(F03_IN, 'audio_clean.mp3')),
    'images/':         os.path.isdir(os.path.join(F03_IN, 'images')),
}
all_ok = True
for name, ok in checks.items():
    status = '✓' if ok else '✗ MANQUANT'
    print(f'  {name}: {status}')
    if not ok:
        all_ok = False

if not all_ok:
    print('\n[STOP] Fichiers manquants dans F03/IN/. Corrigez avant de continuer.')
else:
    print('\n[OK] Tous les assets présents.')

---
## Étape 6 — Validation CUSTOS check-out (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(SCRIPTS_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-out', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-out FAIL. Vérifiez les fichiers dans F03/IN/.')

---
## Étape 7 — Upload assets vers Modal Volume

> Lit les assets depuis Drive et les envoie dans le Volume Modal.  
> **Première exécution : ~1-2 min selon la taille des images.**  
> Les exécutions suivantes sont plus rapides (Volume déjà peuplé).

In [ ]:
import os, sys, asyncio, threading
sys.path.insert(0, SCRIPTS_DIR)
import modal
from crs_f03_modal_worker import app, upload_assets

def load_assets(f03_in):
    assets = {}
    for fname in ['timing.json', 'roadmap.json', 'audio_clean.mp3']:
        path = os.path.join(f03_in, fname)
        with open(path, 'rb') as f:
            assets[fname] = f.read()
    images_dir = os.path.join(f03_in, 'images')
    if os.path.isdir(images_dir):
        for img_name in sorted(os.listdir(images_dir)):
            img_path = os.path.join(images_dir, img_name)
            if os.path.isfile(img_path):
                with open(img_path, 'rb') as f:
                    assets[f'images/{img_name}'] = f.read()
    return assets

print('[UPLOAD] Lecture des assets depuis Drive...')
assets = load_assets(F03_IN)
total_mb = sum(len(v) for v in assets.values()) / 1024 / 1024
print(f'[UPLOAD] {len(assets)} fichier(s) — {total_mb:.1f} MB total')
print('[UPLOAD] Envoi vers Modal Volume...')

# ── Thread isolé (contourne NestedEventLoops de Colab) ────────────────────
_r = {}
def _upload():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        with app.run():
            upload_assets.remote(assets)
        _r['ok'] = True
    except Exception as e:
        _r['error'] = e
    finally:
        loop.close()

t = threading.Thread(target=_upload)
t.start()
t.join()
if 'error' in _r:
    raise _r['error']

print('[OK] Assets disponibles sur le Volume Modal.')

---
## Étape 8a — Deploy & Validate (build image + test 30 frames)

> **Lance `modal deploy` pour builder l'image et la mettre en cache sur Modal.**
> Ensuite effectue un mini-render de 30 frames pour vérifier que Remotion + Chromium fonctionnent.
> **Durée : ~5-8 min (première fois), ~1 min (si image déjà en cache).**
> Si cette étape passe, l'Étape 8b ne crashera pas à l'installation.

In [ ]:
import subprocess, sys, os
sys.path.insert(0, SCRIPTS_DIR)

# ── Deploy : build l'image Modal et la met en cache ───────────────────────
print('[8a] Déploiement de l\'app Modal (build image + npm install)...')
worker_path = os.path.join(SCRIPTS_DIR, 'crs_f03_modal_worker.py')
result = subprocess.run(
    ['modal', 'deploy', worker_path],
    check=False
)
if result.returncode != 0:
    raise RuntimeError('[STOP] modal deploy échoué — vérifiez les logs ci-dessus.')
print('[8a] Deploy OK — image buildée et mise en cache.\n')

# ── Validate : mini-render 30 frames via app déployée ────────────────────
import modal

print('[8a] Validation via render_validation (app déployée)...')
validate_fn = modal.Function.from_name('crusader-f03-renderer', 'render_validation')
status = validate_fn.remote(COMPOSITION)

if status == 'OK':
    print('[8a] VALIDATION OK — Remotion + Chromium opérationnels.')
    print('[8a] Prêt pour le render de production (Étape 8b).')
else:
    raise RuntimeError('[STOP] Validation échouée — ne pas lancer Étape 8b.')

---
## Étape 8b — Dispatch rendu parallèle (N workers GPU)

> **Durée estimée : ~12-20 min selon N_WORKERS.**  
> Chaque worker reçoit une plage de frames et rend en parallèle sur GPU A10G.
> L'image est déjà buildée (Étape 8a) — démarrage instantané.

In [ ]:
import json, os, sys
sys.path.insert(0, SCRIPTS_DIR)
import modal

# ── Lecture du nombre total de frames ─────────────────────────────────────
with open(os.path.join(F03_IN, 'roadmap.json')) as f:
    roadmap = json.load(f)

total_frames = max(scene['end_frame'] for scene in roadmap['timeline']) + 1
fps = roadmap['meta']['fps']
print(f'[INFO] Total frames : {total_frames} ({total_frames / fps:.1f} sec @ {fps} fps)')

# ── Découpage en chunks ────────────────────────────────────────────────────
chunk_size = total_frames // N_WORKERS
chunks = []
for i in range(N_WORKERS):
    f_from = i * chunk_size
    f_to   = (f_from + chunk_size - 1) if i < N_WORKERS - 1 else (total_frames - 1)
    chunks.append((i, f_from, f_to))
    print(f'  Chunk {i} : frames {f_from} → {f_to} ({f_to - f_from + 1} frames)')

# ── Spawn via app déployée — workers indépendants de Colab ────────────────
print(f'\n[DISPATCH] Spawn de {N_WORKERS} workers Modal (app déployée)...')
render_chunk_fn = modal.Function.from_name('crusader-f03-renderer', 'render_chunk')
for cid, ff, ft in chunks:
    call = render_chunk_fn.spawn(cid, ff, ft, COMPOSITION)
    print(f'  [SPAWN] Chunk {cid} lancé (frames {ff}→{ft})')

print()
print('[OK] Workers lancés — ils tournent indépendamment de Colab.')
print('[INFO] Tu peux fermer Colab. Reviens dans ~18 min et lance Étape 8b.')


---
## Étape 8c — Récupération des chunks depuis le Volume Modal

> Relance cette cellule après être revenu sur Colab.  
> Elle attend que les 3 chunks soient prêts, puis les télécharge.
> **Si Colab a crashé pendant l'Étape 8b**, relance directement cette cellule (étapes 1→3 d'abord).

In [ ]:
import time, sys, os
sys.path.insert(0, SCRIPTS_DIR)
import modal
from crs_f03_modal_worker import app, list_ready_chunks, get_chunk_from_volume

print(f'[8b] Vérification des chunks dans le Volume Modal...')
print(f'[INFO] En attente de {N_WORKERS} chunks...\n')

POLL_INTERVAL = 30  # secondes entre chaque vérification

with app.run():
    # ── Poll jusqu'à ce que tous les chunks soient prêts ──────────────────
    while True:
        ready = list_ready_chunks.remote(N_WORKERS)
        pending = [i for i in range(N_WORKERS) if i not in ready]
        print(f'  Chunks prêts : {ready}  |  En attente : {pending}')
        if len(ready) == N_WORKERS:
            break
        print(f'  → Prochain check dans {POLL_INTERVAL}s...')
        time.sleep(POLL_INTERVAL)

    # ── Téléchargement ────────────────────────────────────────────────────
    print(f'\n[DOWNLOAD] Récupération des {N_WORKERS} chunks...')
    chunks_data = []
    for cid in range(N_WORKERS):
        data = get_chunk_from_volume.remote(cid)
        chunks_data.append(data)
        print(f'  [OK] chunk_{cid:03d}.mp4 — {len(data)/1024/1024:.1f} MB')

total_mb = sum(len(d) for d in chunks_data) / 1024 / 1024
print(f'\n[OK] {len(chunks_data)} chunks récupérés — {total_mb:.1f} MB total')
print('[INFO] Lance maintenant Étape 9 (concat).')


---
## Étape 9 — Concat chunks → short_render.mp4

In [ ]:
import sys, asyncio, threading
sys.path.insert(0, SCRIPTS_DIR)
import modal
from crs_f03_modal_worker import app, concat_chunks

print('[CONCAT] Assemblage des chunks...')

# ── Thread isolé (contourne NestedEventLoops de Colab) ────────────────────
_r = {}
def _concat():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        with app.run():
            _r['data'] = concat_chunks.remote(chunks_data)
    except Exception as e:
        _r['error'] = e
    finally:
        loop.close()

t = threading.Thread(target=_concat)
t.start()
t.join()
if 'error' in _r:
    raise _r['error']

final_video_bytes = _r['data']
size_mb = len(final_video_bytes) / 1024 / 1024
print(f'[OK] short_render.mp4 — {size_mb:.1f} MB')

---
## Étape 10 — Sauvegarde sur Drive (F03/OUT/)

In [ ]:
import os

output_path = os.path.join(F03_OUT, 'short_render.mp4')
os.makedirs(F03_OUT, exist_ok=True)

with open(output_path, 'wb') as f:
    f.write(final_video_bytes)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'[OK] Vidéo sauvegardée → {output_path}')
print(f'     Taille : {size_mb:.1f} MB')

---
## Étape 11 — Validation CUSTOS check-in (F03)

In [ ]:
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, os.path.join(SCRIPTS_DIR, 'CRS_CUSTOS.py'),
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
)
if result.returncode != 0:
    print('[STOP] CUSTOS check-in FAIL. short_render.mp4 absent ou trop petit.')
else:
    print('[OK] short_render.mp4 validé — prêt pour transfert vers F04.')

---
## Étape 12 — Aperçu et téléchargement

In [ ]:
import os
from IPython.display import Video, display

output_path = os.path.join(F03_OUT, 'short_render.mp4')

if os.path.isfile(output_path):
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f'short_render.mp4 — {size_mb:.1f} MB')
    print(f'Chemin : {output_path}')
    print()
    display(Video(output_path, embed=True, width=360))
else:
    print('[ERREUR] short_render.mp4 introuvable.')

In [ ]:
# Téléchargement direct depuis Colab (optionnel)
from google.colab import files
files.download(output_path)